In [ ]:
import json
import csv
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

results_file = Path("./outputs/results.csv")

tasks_setup = {
    "rte":  {"lr": 1e-3, "metric": "Accuracy"},
    "mrpc": {"lr": 7e-4, "metric": "F1"},
    "cola": {"lr": 7e-4, "metric": "MCC"},
    "sst2": {"lr": 4e-4, "metric": "Accuracy"},
    "qnli": {"lr": 1e-4, "metric": "Accuracy"},
    "mnli": {"lr": 1e-4, "metric": "Accuracy"},
    "qqp":  {"lr": 4e-4, "metric": "F1"},
    "stsb": {"lr": 1e-4, "metric": "Spearman"}
}
#  bert fullft: str(2e-5)
#  dbert fullft: qnli => 2e-4
#  dbert bitfit rte => 3e-4

In [9]:
def run_task(task, 
             fine_tune_type="bitfit", 
             model="bert-base-cased", 
             seeds=[1, 2, 3, 4, 5], 
             batch=16, 
             gpu=0,
             model_name="bert"):

    config = tasks_setup[task]
    metric = config["metric"]
    results = []

    for seed in seeds:
        output_path = Path("./outputs") / f"{model_name}_{fine_tune_type}_{task}_seed{seed}"
        output_path.mkdir(parents=True, exist_ok=True)

        subprocess.run(["python", "run_glue.py",
                        "--output-path", str(output_path),
                        "--task-name", task,
                        "--model-name", model,
                        "--fine-tune-type", fine_tune_type,
                        "--learning-rate", "3e-4",
                        "--epochs", str(20),
                        "--batch-size", str(batch),
                        "--gpu-device", str(gpu),
                        "--seed", str(seed)])

        with open(output_path / "eval_results.json") as f:
            acc = json.load(f)["validation"][metric]
        results.append(acc)

    mean = np.mean(results) * 100
    std = np.std(results) * 100

    results_file.parent.mkdir(parents=True, exist_ok=True)
    write_header = not results_file.exists()
    with open(results_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["model", "task", "fine_tune_type", "metric", "mean", "std"])
        if write_header:
            writer.writeheader()
        writer.writerow({"model": model, 
                         "task": task, 
                         "fine_tune_type": fine_tune_type,
                         "metric": metric, 
                         "mean": mean, 
                         "std": std})

In [ ]:
!pip install torch transformers datasets tokenizers scikit-learn scipy sentencepiece sacremoses tqdm matplotlib seaborn pandas numpy

In [ ]:
run_task("rte", fine_tune_type="bitfit", model="distilbert-base-cased", model_name="d-bert")

2026-03-16 17:44:54,561 - run_glue - INFO - ############################################################################################
2026-03-16 17:44:54,561 - run_glue - INFO - ############################################################################################
2026-03-16 17:44:54,561 - run_glue - INFO - ############################################################################################
2026-03-16 17:44:54,561 - run_glue - INFO - 
2026-03-16 17:44:54,561 - run_glue - INFO - Training Details: 
2026-03-16 17:44:54,561 - run_glue - INFO - ----------------------------------------------
2026-03-16 17:44:54,561 - run_glue - INFO - Model Name: distilbert-base-cased
2026-03-16 17:44:54,561 - run_glue - INFO - Task Name: rte
2026-03-16 17:44:54,561 - run_glue - INFO - Fine Tuning Type: bitfit
2026-03-16 17:44:54,561 - run_glue - INFO - Output Directory: outputs/d-bert_bitfit_rte_seed1
2026-03-16 17:44:54,561 - run_glue - INFO - Running on GPU #0
2026-03-16 17:44:54,561 - ru

2026-03-16 17:44:55,314 - _client - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/glue/glue.py "HTTP/1.1 200 OK"
2026-03-16 17:44:55,475 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/glue/revision/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c "HTTP/1.1 307 Temporary Redirect"
2026-03-16 17:44:55,572 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/nyu-mll/glue/revision/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c "HTTP/1.1 200 OK"
2026-03-16 17:44:55,674 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
2026-03-16 17:44:55,773 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/nyu-mll/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-03-16 17:44:55,972 - _client - INFO - HTTP Request: GET https://datasets

Map: 4980 examples [00:00, 7102.49 examples/s]         
Map: 554 examples [00:00, 9221.62 examples/s]        
Map: 6000 examples [00:00, 7688.36 examples/s]         


2026-03-16 17:45:00,998 - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6588.91it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2807.62it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.




Trainable Components:
----------------------------------------

distilbert.embeddings.LayerNorm.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.q_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.k_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.v_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.out_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.sa_layer_norm.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.ffn.lin1.bias   --->   torch.Size([3072])
distilbert.transformer.layer.0.ffn.lin2.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.output_layer_norm.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.q_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.k_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.v_lin.bias   --->   torch.Size([768])
distilbert.t

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5948.61it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-03-16 18:12:55,837 - run_glue - INFO - ############################################################################################
2026-03-16 18:12:55,837 - run_glue - INFO - ############################################################################################
2026-03-16 18:12:55,837 - run_glue - INFO - ############################################################################################
2026-03-16 18:12:55,837 - run_glue - INFO - 
2026-03-16 18:12:55,837 - run_glue - INFO - Training Details: 
2026-03-16 18:12:55,837 - run_glue - INFO - ----------------------------------------------
2026-03-16 18:12:55,837 - run_glue - INFO - Model Name: distilbert-base-cased
2026-03-16 18:12:55,837 - run_glue - INFO - Task Name: rte
2026-03-16 18:12:55,837 - run_glue - INFO - Fine Tuning Type: bitfit
2026-03-16 18:12:55,837 - run_glue - INFO - Output Directory: outputs/d-bert_bitfit_rte_seed3
2026-03-16 18:12:55,837 - run_glue - INFO - Running on GPU #0
2026-03-16 18:12:55,837 - ru

Map: 4980 examples [00:00, 9578.69 examples/s]         
Map: 554 examples [00:00, 7946.75 examples/s]        
Map: 6000 examples [00:00, 8784.50 examples/s]         
